# Modular Neuro-Symbolic News Pipeline

**Gemini edition:** Google Gemini extraction, structured JSON parsing,
cross-field validation, temporal event memory, fuzzy predicate grounding, and LTN reasoning.

The notebook deliberately processes a small chronological sample so every transformation
can be inspected before scaling.


## 1. Google Gemini API setup

Run this installation cell once in the notebook environment if needed:

```python
%pip install -U google-genai python-dotenv
```

The notebook uses Gemini through Google AI Studio / Gemini API, so no local LLM
weights are downloaded. Add `GEMINI_API_KEY` to `env.txt` or `.env`; the value is
loaded from the environment and is never printed.


In [18]:
from __future__ import annotations

import hashlib
import json
import os
import re
import time
from collections import deque
from datetime import datetime
from pathlib import Path
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download
from pydantic import BaseModel, Field, ValidationError

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 50)

# Search from the current working directory. In the local dissertation workspace,
# also try the sibling Sentiment project without exposing any secret values.

PROJECT_ROOT = Path("/home/jovyan/Stock-Sentiment-Prediction")
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

dotenv_candidates = [
    PROJECT_ROOT / "env.txt",
    PROJECT_ROOT / ".env",
    *[
        parent / "Sentiment project" / "env.txt"
        for parent in (Path.cwd(), *Path.cwd().parents)
    ],
    *[
        parent / "Sentiment project" / ".env"
        for parent in (Path.cwd(), *Path.cwd().parents)
    ],
]
dotenv_path = next((path for path in dotenv_candidates if path.exists()), None)

if dotenv_path:
    load_dotenv(dotenv_path=str(dotenv_path))

HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN, "HF_TOKEN is missing. Add it to your env.txt or .env file before continuing."

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "GEMINI_API_KEY is missing. Add it to your env.txt or .env file before continuing."

DATASET_REPO = "cookekieran/alphavantage-market-news"
MODEL_ID = os.getenv("GEMINI_MODEL_ID", "gemini-2.5-flash")
MONTH = "2023-01"
TICKER = "AAPL"
MIN_TICKER_RELEVANCE = 0.20
MAX_ARTICLES = 60

# Keep True while learning the pipeline. It creates deterministic placeholder
# assessments so every non-model stage can be inspected cheaply.
USE_MOCK_LLM = False

# Set to 1 while inspecting individual article behaviour. Increase later for throughput.
GEMINI_BATCH_SIZE = int(os.getenv("GEMINI_BATCH_SIZE", "15"))
GEMINI_SLEEP_SECONDS = float(os.getenv("GEMINI_SLEEP_SECONDS", "13"))

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "modular_news_demo" / f"{TICKER}_{MONTH}_{MODEL_ID.replace('/', '_')}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({
    "dataset": DATASET_REPO,
    "model": MODEL_ID,
    "month": MONTH,
    "ticker": TICKER,
    "max_articles": MAX_ARTICLES,
    "use_mock_llm": USE_MOCK_LLM,
    "gemini_batch_size": GEMINI_BATCH_SIZE,
    "output_dir": str(OUTPUT_DIR),
})


{'dataset': 'cookekieran/alphavantage-market-news', 'model': 'gemini-2.5-flash', 'month': '2023-01', 'ticker': 'AAPL', 'max_articles': 60, 'use_mock_llm': False, 'gemini_batch_size': 15, 'output_dir': 'c:\\Users\\kiera\\OneDrive\\University\\Univeristy\\Studying\\Masters\\Dissertation\\Sentiment project\\outputs\\modular_news_demo\\AAPL_2023-01_gemini-2.5-flash'}


## 2. Download and join the January news partitions

The dataset stores:

- `articles`: title, summary, source, timestamp, and stable `article_uid`;
- `tickers`: ticker relevance and Alpha Vantage ticker sentiment;
- `topics`: topic relevance.

We filter through the ticker-link table rather than relying on
`requested_entity`, because an article requested for one entity may still be
relevant to another.

In [2]:
def download_partition(folder: str) -> Path:
    return Path(
        hf_hub_download(
            repo_id=DATASET_REPO,
            repo_type="dataset",
            filename=f"{folder}/{MONTH}.parquet",
            token=HF_TOKEN,
        )
    )


articles = pd.read_parquet(download_partition("articles"))
tickers = pd.read_parquet(download_partition("tickers"))
topics = pd.read_parquet(download_partition("topics"))

print("articles:", articles.shape)
print("tickers:", tickers.shape)
print("topics:", topics.shape)
display(articles.head(2))

articles: (2344, 11)
tickers: (4220, 7)
topics: (6254, 5)


,article_id,article_uid,time_published,overall_sentiment_score,overall_sentiment_label,title,summary,source,url,requested_entity,month
0,0,cc943125f2c299c334e9484c073748f4,2023-01-02 04:52:00,0.269256,Somewhat-Bullish,What is TSMC?,"Taiwan Semiconductor Manufacturing Co (TSMC) is the world's largest semiconductor manufacturer, producing chips for major tech companies...",Tech Monitor,https://www.techmonitor.ai/what-is/what-is-tsmc/,AAPL,2023-01
1,1,58d72b86288d4fffa86b70eac0d65e89,2023-01-03 03:59:02,-0.175983,Somewhat-Bearish,Apple’s market value drops below $2 trillion,"Apple Inc.'s market capitalization fell below $2 trillion for the first time since March 2021, driven by concerns over high inflation an...",Al Jazeera,https://www.aljazeera.com/economy/2023/1/3/apples-market-value-drops-below-2-trillion,MSFT,2023-01


In [3]:
ticker_links = (
    tickers.loc[
        (tickers["ticker"] == TICKER)
        & (tickers["relevance_score"] >= MIN_TICKER_RELEVANCE)
    ]
    .rename(
        columns={
            "relevance_score": "ticker_relevance",
            "ticker_sentiment_score": "av_ticker_sentiment_score",
            "ticker_sentiment_label": "av_ticker_sentiment_label",
        }
    )
    [
        [
            "article_uid",
            "ticker",
            "ticker_relevance",
            "av_ticker_sentiment_score",
            "av_ticker_sentiment_label",
        ]
    ]
)

topic_lists = (
    topics.sort_values(["article_uid", "relevance_score"], ascending=[True, False])
    .groupby("article_uid")
    .apply(
        lambda frame: [
            {"topic": row.topic, "relevance": float(row.relevance_score)}
            for row in frame.itertuples()
        ],
        include_groups=False,
    )
    .rename("topics")
    .reset_index()
)

article_stream = (
    articles.merge(ticker_links, on="article_uid", how="inner", validate="one_to_one")
    .merge(topic_lists, on="article_uid", how="left", validate="one_to_one")
    .drop_duplicates("article_uid")
    .sort_values(["time_published", "article_uid"])
    .head(MAX_ARTICLES)
    .reset_index(drop=True)
)
article_stream["topics"] = article_stream["topics"].apply(
    lambda value: value if isinstance(value, list) else []
)

print(f"Selected {len(article_stream)} chronological {TICKER} articles.")
display(
    article_stream[
        [
            "time_published",
            "title",
            "source",
            "ticker_relevance",
            "av_ticker_sentiment_label",
        ]
    ]
)

Selected 10 chronological AAPL articles.


,time_published,title,source,ticker_relevance,av_ticker_sentiment_label
0,2023-01-02 04:52:00,What is TSMC?,Tech Monitor,0.718922,Somewhat-Bullish
1,2023-01-03 03:59:02,Apple’s market value drops below $2 trillion,Al Jazeera,1.000000,Bearish
2,2023-01-03 04:06:10,"If You Invested $100 in Berkshire Hathaway in 1965, This Is How Much You Would Have Today",The Motley Fool,0.732629,Somewhat-Bullish
3,2023-01-03 04:06:10,"If You Invested $100 in Berkshire Hathaway in 1965, This Is How Much You Would Have Today",The Globe and Mail,0.749384,Somewhat-Bullish
4,2023-01-03 06:49:00,Foxconn to use Nvidia chips to build self-driving vehicle platforms,Reuters,0.622889,Neutral
5,2023-01-03 12:05:00,Tuesday's ETF with Unusual Volume: IWL,Nasdaq,0.784926,Somewhat-Bearish
6,2023-01-03 17:38:38,Think you have it bad?,The Reformed Broker,0.714321,Somewhat-Bearish
7,2023-01-04 09:26:00,Salesforce to lay off 10% of workforce to cut costs amid economic downturn,Fox Business,0.584422,Somewhat-Bearish
8,2023-01-04 11:52:00,Apple to sign Luxshare for iPhone production in China - FT,Reuters,1.000000,Somewhat-Bullish
9,2023-01-05 03:59:02,Apple brings AI narration to audiobooks,SiliconANGLE,0.981440,Bullish


## 3. Define the LLM's structured output

The LLM returns **ordinal event attributes**, not unexplained decimal scores.
Every category has a stable definition in the prompt.

Event types and transmission channels are intentionally broader than individual
historical events. For example, an article about a specific conflict may create
an event called `Taiwan semiconductor conflict risk`, while its reusable
attributes include:

```text
event_type = geopolitical_conflict
transmission_channels = [supply_chain, operational_disruption]
```

Later, those attributes ground stable predicates such as
`GeopoliticalRisk(AAPL, date)` and `SupplyChainRisk(AAPL, date)`.

In [4]:
Level = Literal["none", "low", "medium", "high", "very_high"]
Direction = Literal["risk_on", "risk_off", "mixed", "neutral"]
Scope = Literal["ticker", "sector", "macro", "broad_market", "irrelevant"]
EventAction = Literal[
    "new",
    "confirm",
    "escalate",
    "deescalate",
    "contradict",
    "repeat",
    "resolve",
    "irrelevant",
]
EventType = Literal[
    "earnings_outlook",
    "product_or_innovation",
    "demand_change",
    "supply_chain",
    "regulatory_or_legal",
    "geopolitical_conflict",
    "management_or_strategy",
    "financing_or_balance_sheet",
    "competitive_position",
    "operational_disruption",
    "macroeconomic",
    "market_sentiment",
    "other",
]
TransmissionChannel = Literal[
    "revenue_growth",
    "demand_weakening",
    "margin_pressure",
    "input_cost_pressure",
    "financing_conditions",
    "supply_chain",
    "operational_disruption",
    "regulatory_restriction",
    "competitive_position",
    "market_uncertainty",
    "none",
]


class ArticleEventAssessment(BaseModel):
    article_uid: str
    ticker: str
    relevant: bool
    matched_event_id: str | None = None
    event_action: EventAction
    event_name: str
    event_type: EventType
    event_summary: str
    direction: Direction
    scope: Scope
    transmission_channels: list[TransmissionChannel]
    materiality: Level
    novelty: Level
    confidence: Level
    evidence_summary: str


LEVEL_VALUE = {
    "none": 0.0,
    "low": 0.25,
    "medium": 0.50,
    "high": 0.75,
    "very_high": 1.0,
}


def model_schema(model_class: type[BaseModel]) -> dict:
    # Support both Pydantic v1 and v2.
    if hasattr(model_class, "model_json_schema"):
        return model_class.model_json_schema()
    return model_class.schema()


def model_dump_compat(model: BaseModel) -> dict:
    # Support both Pydantic v1 and v2.
    if hasattr(model, "model_dump"):
        return model.model_dump()
    return model.dict()


def model_validate_json_compat(model_class: type[BaseModel], payload: str) -> BaseModel:
    # Support both Pydantic v1 and v2.
    if hasattr(model_class, "model_validate_json"):
        return model_class.model_validate_json(payload)
    return model_class.parse_raw(payload)

## 4. Temporal event-memory tables

The LLM has no persistent memory between calls. Before each article, Python
supplies a compact context packet containing:

- candidate active events created by earlier articles;
- recent article assessments;
- the current article.

After validation, Python updates three auditable temporal tables:

1. `events`: current event records;
2. `event_state_history`: every historical state change;
3. `event_evidence`: which article caused each change.

Because articles are processed chronologically, a historical call cannot see a
future event update.

In [5]:
def stable_event_id(ticker: str, event_type: str, event_name: str) -> str:
    digest = hashlib.sha1(event_name.lower().encode("utf-8")).hexdigest()[:10]
    return f"{ticker.lower()}__{event_type}__{digest}"

def max_level(a: str, b: str) -> str:
    return max([a, b], key=lambda level: LEVEL_VALUE[level])


def bump_confidence(previous: str, new: str, action: str) -> str:
    previous_value = LEVEL_VALUE.get(previous, 0.0)
    new_value = LEVEL_VALUE.get(new, 0.0)

    if action == "repeat":
        # Repetition gives only a small confidence bump.
        combined = min(1.0, max(previous_value, new_value) + 0.05)
    elif action == "confirm":
        # New supporting evidence gives a stronger bump.
        combined = min(1.0, max(previous_value, new_value) + 0.15)
    else:
        combined = max(previous_value, new_value)

    return min(LEVEL_VALUE, key=lambda level: abs(LEVEL_VALUE[level] - combined))


class TemporalEventMemory:
    def __init__(self):
        self.events: dict[str, dict] = {}
        self.state_history: list[dict] = []
        self.evidence: list[dict] = []
        self.recent_assessments = deque(maxlen=6)

    def candidate_events(self, ticker: str, limit: int = 8) -> list[dict]:
        active = [
            event
            for event in self.events.values()
            if event["status"] != "resolved"
            and (event["ticker"] == ticker or event["scope"] in {"sector", "macro", "broad_market"})
        ]
        active.sort(key=lambda event: event["last_updated"], reverse=True)
        return active[:limit]

    def context_packet(self, article: pd.Series) -> dict:
        return {
            "processing_timestamp": pd.Timestamp(article.time_published).isoformat(),
            "ticker": article.ticker,
            "article": {
                "article_uid": article.article_uid,
                "published_at": pd.Timestamp(article.time_published).isoformat(),
                "title": article.title,
                "summary": article.summary,
                "source": article.source,
                "ticker_relevance": float(article.ticker_relevance),
                "topics": article.topics,
            },
            "candidate_active_events": self.candidate_events(article.ticker),
            "recent_assessments": list(self.recent_assessments),
        }

    def apply(self, assessment: ArticleEventAssessment, observed_at: pd.Timestamp) -> str | None:
        if not assessment.relevant or assessment.event_action == "irrelevant":
            self.recent_assessments.append(model_dump_compat(assessment))
            return None

        event_id = assessment.matched_event_id
        if event_id not in self.events:
            event_id = stable_event_id(
                assessment.ticker,
                assessment.event_type,
                assessment.event_name,
            )

        previous = self.events.get(event_id, {})
        status = {
            "new": "unresolved",
            "confirm": previous.get("status", "unresolved"),
            "repeat": previous.get("status", "unresolved"),
            "escalate": "escalating",
            "deescalate": "deescalating",
            "contradict": "contested",
            "resolve": "resolved",
        }[assessment.event_action]

        event = {
            "event_id": event_id,
            "ticker": assessment.ticker,
            "event_name": assessment.event_name,
            "event_type": assessment.event_type,
            "summary": assessment.event_summary,
            "scope": assessment.scope,
            "direction": assessment.direction,
            "transmission_channels": assessment.transmission_channels,
            
            "materiality": max_level(
                previous.get("materiality", assessment.materiality),
                assessment.materiality),
            
            "confidence": bump_confidence(
                previous.get("confidence", assessment.confidence),
                assessment.confidence,
                assessment.event_action),
            
            "status": status,
            "first_seen": previous.get("first_seen", observed_at.isoformat()),
            "last_updated": observed_at.isoformat(),
        }
        self.events[event_id] = event
        self.state_history.append(
            {
                **event,
                "valid_from": observed_at.isoformat(),
                "event_action": assessment.event_action,
            }
        )
        self.evidence.append(
            {
                "event_id": event_id,
                "article_uid": assessment.article_uid,
                "observed_at": observed_at.isoformat(),
                "event_action": assessment.event_action,
                "evidence_summary": assessment.evidence_summary,
            }
        )
        recent = model_dump_compat(assessment)
        recent["resolved_event_id"] = event_id
        self.recent_assessments.append(recent)
        return event_id

    def tables(self) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        return (
            pd.DataFrame(self.events.values()),
            pd.DataFrame(self.state_history),
            pd.DataFrame(self.evidence),
        )

In [6]:
# Inspect exactly what the first model call can see.
preview_memory = TemporalEventMemory()
first_context_packet = preview_memory.context_packet(article_stream.iloc[0])
print(json.dumps(first_context_packet, indent=2, default=str)[:5000])

{
  "processing_timestamp": "2023-01-02T04:52:00",
  "ticker": "AAPL",
  "article": {
    "article_uid": "cc943125f2c299c334e9484c073748f4",
    "published_at": "2023-01-02T04:52:00",
    "title": "What is TSMC?",
    "summary": "Taiwan Semiconductor Manufacturing Co (TSMC) is the world's largest semiconductor manufacturer, producing chips for major tech companies like Apple, Amazon, and Google. It focuses solely on manufacturing for third parties, unlike competitors, and boasts significant market dominance and advanced process technology. The article also discusses the potential impact of a Chinese invasion of Taiwan on TSMC and its global operations, as well as the company's expansion plans in the US.",
    "source": "Tech Monitor",
    "ticker_relevance": 0.718922,
    "topics": [
      {
        "topic": "technology",
        "relevance": 0.943473
      },
      {
        "topic": "manufacturing",
        "relevance": 0.817952
      },
      {
        "topic": "economy_fiscal",
   

## 5. DeepSeek prompt and structured-response parser

DeepSeek receives all instructions in the user prompt. The prompt explicitly
prohibits regime prediction and asks for a final JSON object between markers.

The raw response is retained before parsing. This allows malformed outputs and
model reasoning to be audited later.

In [7]:
RUBRIC = """
Use these ordinal rubrics consistently:

MATERIALITY (ex ante; do not use later price movement)
- none: no plausible financial consequence
- low: minor commentary or weak operational relevance
- medium: plausible but limited impact on revenue, costs, operations, regulation, or strategy
- high: meaningful impact likely to affect company fundamentals or valuation
- very_high: potentially transformative or severely disruptive impact

NOVELTY (relative only to supplied prior context)
- none: irrelevant or no information
- low: repetition of already-known information
- medium: useful confirmation or modest new detail
- high: important new development
- very_high: unexpected development that substantially changes the known event

CONFIDENCE (confidence in the extracted claim, not confidence in a price forecast)
- none: unusable evidence
- low: speculation or unclear sourcing
- medium: plausible reporting with uncertainty
- high: well-supported reporting
- very_high: direct authoritative confirmation
"""

ALLOWED_EVENT_TYPES = {
    "earnings_outlook", "product_or_innovation", "demand_change", "supply_chain",
    "regulatory_or_legal", "geopolitical_conflict", "management_or_strategy",
    "financing_or_balance_sheet", "competitive_position", "operational_disruption",
    "macroeconomic", "market_sentiment", "other",
}
ALLOWED_ACTIONS = {
    "new", "confirm", "escalate", "deescalate", "contradict", "repeat",
    "resolve", "irrelevant",
}
ALLOWED_SCOPES = {"ticker", "sector", "macro", "broad_market", "irrelevant"}
ALLOWED_DIRECTIONS = {"risk_on", "risk_off", "mixed", "neutral"}
ALLOWED_CHANNELS = {
    "revenue_growth", "demand_weakening", "margin_pressure", "input_cost_pressure",
    "financing_conditions", "supply_chain", "operational_disruption",
    "regulatory_restriction", "competitive_position", "market_uncertainty", "none",
}
ALLOWED_LEVELS = set(LEVEL_VALUE)


def build_prompt(context_packet: dict) -> str:
    schema = model_schema(ArticleEventAssessment)
    return f"""
You are an event analyst for a modular neuro-symbolic financial-news system.

Analyse only the supplied article using only the supplied prior context. Do not
predict a market regime, stock return, or price direction. Extract one event
assessment describing reusable economic attributes.

DECISION ORDER
1. Decide whether the article is directly relevant to the target ticker.
2. If irrelevant, return the exact irrelevant template below.
3. If relevant, compare it with candidate_active_events.
4. Match an existing event only when the real-world development, main actors,
   subject, and causal mechanism are the same.
5. Otherwise create a new event.

RELEVANCE
- Relevant means the article describes a development with a direct causal link
  to the ticker's operations, revenue, costs, competitive position, strategy,
  financing, or regulatory environment.
- Passive mentions, generic market commentary, and competitor news without a
  described direct consequence for the ticker are irrelevant.

HARD EVIDENCE GATE FOR RELEVANCE

Before setting relevant=true, identify the exact sentence or claim in the article
that creates a direct causal link to the target ticker.

If the article only mentions the ticker in any of the following ways, set
relevant=false:

- as a member of a stock index, ETF, portfolio, or basket
- as one example in a list of companies
- as a peer/comparison company
- as background context
- as part of broad technology-sector, macroeconomic, labour-market, or market-sentiment commentary
- as an investment recommendation without a new company-specific development
- in a consumer review, shopping guide, product ranking, or generic comparison article

For relevant=true, the evidence_summary must explicitly name the target ticker
and the concrete business mechanism affected.

Bad evidence_summary:
"The article discusses weakness in the tech sector."

Good evidence_summary:
"The article states Apple iPhone production is shifting to Luxshare, directly
affecting Apple's supplier base for premium iPhone models."

IRRELEVANT TEMPLATE
- relevant: false
- matched_event_id: null
- event_action: "irrelevant"
- event_name: "No material event"
- event_type: "other"
- event_summary: "No material event identified for the target ticker."
- direction: "neutral"
- scope: "irrelevant"
- transmission_channels: ["none"]
- materiality: "none"
- novelty: "none"

EVENT MATCHING
- Candidate events and recent assessments are the complete allowed history.
- Do not assume future knowledge.
- Same ticker, direction, sector, or transmission channel is insufficient.
- Use repeat for substantially duplicate reporting with no meaningful new fact.
- Use confirm only for new supporting evidence about the same existing event.
- Use escalate/deescalate/contradict/resolve only for an explicit change.
- matched_event_id must be copied exactly from candidate_active_events.
- For event_action="new" or "irrelevant", matched_event_id must be null.

{RUBRIC}

OUTPUT REQUIREMENTS
- Return exactly one JSON object between <event_json> and </event_json>.
- Do not include reasoning, commentary, markdown fences, or a second object.
- Use only enum values from the schema.
- Copy article_uid and ticker exactly from the context packet.
- Only matched_event_id may be null.
- If evidence_summary does not contain a concrete target-ticker mechanism, set
relevant=false.


JSON schema:
{json.dumps(schema, indent=2)}

Context packet:
{json.dumps(context_packet, indent=2, default=str)}
""".strip()


def normalise_enum(value: object) -> str:
    return str(value).strip().lower().replace("-", "_").replace(" ", "_")


def normalise_bool(value: object) -> bool:
    if isinstance(value, bool):
        return value
    return normalise_enum(value) in {"true", "1", "yes"}


def sanitise_assessment_payload(data: dict, context_packet: dict) -> dict:
    """Repair formatting errors and enforce cross-field invariants."""
    repaired = dict(data)
    article = context_packet["article"]
    candidate_ids = {
        event["event_id"] for event in context_packet["candidate_active_events"]
    }

    repaired["article_uid"] = article["article_uid"]
    repaired["ticker"] = context_packet["ticker"]
    repaired["relevant"] = normalise_bool(repaired.get("relevant", False))

    repaired["event_type"] = normalise_enum(repaired.get("event_type", "other"))
    if repaired["event_type"] not in ALLOWED_EVENT_TYPES:
        repaired["event_type"] = "other"

    repaired["direction"] = normalise_enum(repaired.get("direction", "neutral"))
    if repaired["direction"] not in ALLOWED_DIRECTIONS:
        repaired["direction"] = "neutral"

    repaired["scope"] = normalise_enum(repaired.get("scope", "irrelevant"))
    if repaired["scope"] not in ALLOWED_SCOPES:
        repaired["scope"] = "ticker" if repaired["relevant"] else "irrelevant"

    repaired["event_action"] = normalise_enum(
        repaired.get("event_action", "new" if repaired["relevant"] else "irrelevant")
    )
    if repaired["event_action"] not in ALLOWED_ACTIONS:
        repaired["event_action"] = "new" if repaired["relevant"] else "irrelevant"

    channels = repaired.get("transmission_channels", ["none"])
    channels = channels if isinstance(channels, list) else [channels]
    channels = [normalise_enum(channel) for channel in channels]
    repaired["transmission_channels"] = list(
        dict.fromkeys(channel for channel in channels if channel in ALLOWED_CHANNELS)
    ) or ["none"]

    for field in ("materiality", "novelty", "confidence"):
        repaired[field] = normalise_enum(repaired.get(field, "none"))
        if repaired[field] not in ALLOWED_LEVELS:
            repaired[field] = "none"

    if not repaired["relevant"] or repaired["event_action"] == "irrelevant":
        repaired.update({
            "relevant": False,
            "matched_event_id": None,
            "event_action": "irrelevant",
            "event_name": "No material event",
            "event_type": "other",
            "event_summary": "No material event identified for the target ticker.",
            "direction": "neutral",
            "scope": "irrelevant",
            "transmission_channels": ["none"],
            "materiality": "none",
            "novelty": "none",
        })
    else:
        matched_id = repaired.get("matched_event_id")
        if repaired["event_action"] == "new" or matched_id not in candidate_ids:
            repaired["event_action"] = "new"
            repaired["matched_event_id"] = None

    fallbacks = {
        "event_name": article.get("title") or "Unnamed event",
        "event_summary": article.get("summary") or "No summary supplied.",
        "evidence_summary": "No evidence summary supplied.",
    }
    for field, fallback in fallbacks.items():
        value = repaired.get(field)
        if not isinstance(value, str) or not value.strip():
            repaired[field] = fallback

    return repaired


def json_objects(text: str) -> list[dict]:
    """Find complete JSON objects even when the model adds surrounding prose."""
    decoder = json.JSONDecoder()
    objects = []
    for index, character in enumerate(text):
        if character != "{":
            continue
        try:
            value, _ = decoder.raw_decode(text[index:])
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict):
            objects.append(value)
    return objects


def extract_event_json(raw_text: str, context_packet: dict) -> ArticleEventAssessment:
    tagged = re.findall(
        r"<event_json>\s*(.*?)\s*</event_json>",
        raw_text,
        flags=re.S | re.I,
    )

    search_areas = []
    for area in tagged:
        cleaned = area.strip()

        # Qwen sometimes returns JSON fields without the outer braces.
        if not cleaned.startswith("{") and '"article_uid"' in cleaned:
            cleaned = "{" + cleaned.rstrip().rstrip(",") + "}"

        search_areas.append(cleaned)

    search_areas.append(raw_text)

    candidates = [
        candidate
        for area in search_areas
        for candidate in json_objects(area)
        if "relevant" in candidate or "event_action" in candidate
    ]

    if not candidates:
        raise ValueError("No assessment JSON object found in the model response")

    repaired = sanitise_assessment_payload(candidates[-1], context_packet)
    return model_validate_json_compat(
        ArticleEventAssessment,
        json.dumps(repaired),
    )

In [9]:
if not USE_MOCK_LLM:
    from google import genai
    from google.genai import types

    gemini_client = genai.Client(api_key=GEMINI_API_KEY)


class ArticleEventAssessmentBatch(BaseModel):
    assessments: list[ArticleEventAssessment] = Field(
        description="One event assessment for each supplied context packet."
    )


def build_batch_prompt(context_packets: list[dict]) -> str:
    schema = model_schema(ArticleEventAssessmentBatch)
    return f"""
You are an event analyst for a modular neuro-symbolic financial-news system.

Analyse each supplied article independently using only its own supplied prior
context. Do not predict a market regime, stock return, or price direction.
Return one assessment per context packet, in the same order as the input.

For every assessment, follow the same rules as the single-article prompt:

{RUBRIC}

RELEVANCE
- Relevant means the article describes a development with a direct causal link
  to the ticker's operations, revenue, costs, competitive position, strategy,
  financing, or regulatory environment.
- Passive mentions, generic market commentary, and competitor news without a
  described direct consequence for the ticker are irrelevant.
- Before setting relevant=true, identify the exact sentence or claim in the
  article that creates a direct causal link to the target ticker.
- If evidence_summary does not contain a concrete target-ticker mechanism, set
  relevant=false.

IRRELEVANT TEMPLATE
- relevant: false
- matched_event_id: null
- event_action: "irrelevant"
- event_name: "No material event"
- event_type: "other"
- event_summary: "No material event identified for the target ticker."
- direction: "neutral"
- scope: "irrelevant"
- transmission_channels: ["none"]
- materiality: "none"
- novelty: "none"

EVENT MATCHING
- Candidate events and recent assessments in each context packet are the
  complete allowed history for that article.
- Do not assume future knowledge.
- Same ticker, direction, sector, or transmission channel is insufficient.
- matched_event_id must be copied exactly from that article's
  candidate_active_events.
- For event_action="new" or "irrelevant", matched_event_id must be null.

OUTPUT REQUIREMENTS
- Return exactly one JSON object.
- The object must contain an "assessments" array.
- The array must contain exactly {len(context_packets)} items.
- Use only enum values from the schema.
- Copy article_uid and ticker exactly from each context packet.
- Only matched_event_id may be null.

JSON schema:
{json.dumps(schema, indent=2)}

Context packets:
{json.dumps(context_packets, indent=2, default=str)}
""".strip()


def gemini_generate_json(prompt: str, response_schema: dict) -> str:
    response = gemini_client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json",
            response_json_schema=response_schema,
        ),
    )
    return response.text or ""


def generate_model_response(context_packet: dict) -> str:
    prompt = build_prompt(context_packet)
    return gemini_generate_json(prompt, model_schema(ArticleEventAssessment))


def generate_model_responses(context_packets: list[dict]) -> list[str]:
    if len(context_packets) == 1:
        return [generate_model_response(context_packets[0])]

    raw_text = gemini_generate_json(
        build_batch_prompt(context_packets),
        model_schema(ArticleEventAssessmentBatch),
    )
    batch = ArticleEventAssessmentBatch.model_validate_json(raw_text)
    by_article_uid = {
        assessment.article_uid: assessment
        for assessment in batch.assessments
    }

    raw_texts = []
    for context_packet in context_packets:
        article_uid = context_packet["article"]["article_uid"]
        assessment = by_article_uid.get(article_uid)
        if assessment is None:
            raise ValueError(f"Gemini batch response omitted article_uid={article_uid}")
        repaired = sanitise_assessment_payload(
            model_dump_compat(assessment),
            context_packet,
        )
        raw_texts.append(json.dumps(repaired))
    return raw_texts


## 6. Cheap mock mode for understanding the pipeline

Mock mode is deliberately simplistic and must **not** be used as research
output. It lets you inspect event tables, predicate grounding, and LTN reasoning
before spending GPU time.

Set `USE_MOCK_LLM = False` to replace it with DeepSeek.

In [10]:
def mock_assessment(article: pd.Series) -> ArticleEventAssessment:
    score = float(article.av_ticker_sentiment_score)
    direction = "risk_on" if score > 0.15 else "risk_off" if score < -0.15 else "neutral"
    magnitude = abs(score)
    materiality = "high" if magnitude > 0.35 else "medium" if magnitude > 0.15 else "low"
    title_lower = article.title.lower()

    if any(term in title_lower for term in ["supply", "chip", "manufactur"]):
        event_type = "supply_chain"
        channels = ["supply_chain"]
    elif any(term in title_lower for term in ["demand", "sales", "consumer"]):
        event_type = "demand_change"
        channels = ["demand_weakening"] if direction == "risk_off" else ["revenue_growth"]
    elif any(term in title_lower for term in ["market", "stock", "value"]):
        event_type = "market_sentiment"
        channels = ["market_uncertainty"]
    else:
        event_type = "other"
        channels = ["none"]

    return ArticleEventAssessment(
        article_uid=article.article_uid,
        ticker=article.ticker,
        relevant=True,
        matched_event_id=None,
        event_action="new",
        event_name=article.title[:100],
        event_type=event_type,
        event_summary=article.summary[:300],
        direction=direction,
        scope="ticker",
        transmission_channels=channels,
        materiality=materiality,
        novelty="medium",
        confidence="medium",
        evidence_summary="Mock assessment based on existing dataset metadata.",
    )

## 7. Process articles chronologically and build the graph

With `GEMINI_BATCH_SIZE = 1`, each call receives only events and assessments
created before the current article timestamp. Larger batches reduce request count
for big runs, but articles inside the same batch share the same pre-batch memory
state.


In [21]:
memory = TemporalEventMemory()
assessment_records = []
raw_response_records = []
context_records = []

articles = list(article_stream.iterrows())
batch_size = max(1, GEMINI_BATCH_SIZE)

for start in range(0, len(articles), batch_size):
    batch = articles[start:start + batch_size]
    context_packets = []
    batch_articles = []

    for _, article in batch:
        context_packet = memory.context_packet(article)
        context_packets.append(context_packet)
        context_records.append(context_packet)
        batch_articles.append(article)

    try:
        if USE_MOCK_LLM:
            raw_texts = ["MOCK MODE" for _ in batch_articles]
            assessments = [mock_assessment(article) for article in batch_articles]
        else:
            raw_texts = generate_model_responses(context_packets)
            assessments = [
                extract_event_json(raw_text, context_packet)
                for raw_text, context_packet in zip(raw_texts, context_packets)
            ]
    except Exception as exc:
        raw_text = raw_texts[0] if "raw_texts" in locals() and raw_texts else ""
        for article in batch_articles:
            raw_response_records.append({
                "article_uid": article.article_uid,
                "time_published": pd.Timestamp(article.time_published).isoformat(),
                "raw_response": raw_text,
                "error": f"{type(exc).__name__}: {exc}",
            })
            print(f"Skipped {article.article_uid}: {type(exc).__name__}: {exc}")
    else:
        for article, assessment, raw_text in zip(batch_articles, assessments, raw_texts):
            raw_response_records.append({
                "article_uid": article.article_uid,
                "time_published": pd.Timestamp(article.time_published).isoformat(),
                "raw_response": raw_text,
                "error": None,
            })

            event_id = memory.apply(assessment, pd.Timestamp(article.time_published))

            assessment_records.append({
                **model_dump_compat(assessment),
                "resolved_event_id": event_id,
                "time_published": pd.Timestamp(article.time_published).isoformat(),

                # Raw article fields for manual inspection
                "article_title": article.title,
                "article_source": article.source,
                "alphavantage_ticker_relevance": article.ticker_relevance,
                "alphavantage_ticker_sentiment_score": article.av_ticker_sentiment_score,
                "alphavantage_ticker_sentiment_label": article.av_ticker_sentiment_label,
            })

    # Save an auditable checkpoint after every batch, including failures.
    pd.DataFrame(assessment_records).to_json(
        OUTPUT_DIR / "article_assessments.jsonl", orient="records", lines=True,
    )
    pd.DataFrame(raw_response_records).to_json(
        OUTPUT_DIR / "raw_model_responses.jsonl", orient="records", lines=True,
    )
    pd.DataFrame(context_records).to_json(
        OUTPUT_DIR / "context_packets.jsonl",
        orient="records",
        lines=True,
        default_handler=str,
    )
    events_df, states_df, evidence_df = memory.tables()
    events_df.to_parquet(OUTPUT_DIR / "events_current.parquet", index=False)
    states_df.to_parquet(OUTPUT_DIR / "event_state_history.parquet", index=False)
    evidence_df.to_parquet(OUTPUT_DIR / "event_evidence.parquet", index=False)

    if not USE_MOCK_LLM and GEMINI_SLEEP_SECONDS > 0 and start + batch_size < len(articles):
        time.sleep(GEMINI_SLEEP_SECONDS)

assessments_df = pd.DataFrame(assessment_records)
events_df, states_df, evidence_df = memory.tables()
print(f"Processed {len(assessment_records)} of {len(article_stream)} articles.")
display(assessments_df.head(10))


Skipped cc943125f2c299c334e9484c073748f4: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 12.053205998s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location

""


In [17]:
len(assessments_df)

10

In [44]:
display(assessments_df)

,article_uid,ticker,relevant,matched_event_id,event_action,event_name,event_type,event_summary,direction,scope,transmission_channels,materiality,novelty,confidence,evidence_summary,resolved_event_id,time_published,article_title,article_source,alphavantage_ticker_relevance,alphavantage_ticker_sentiment_score,alphavantage_ticker_sentiment_label
0,cc943125f2c299c334e9484c073748f4,AAPL,False,NaN,irrelevant,No material event,other,No material event identified for the target ticker.,neutral,irrelevant,[none],none,none,low,No evidence summary supplied.,NaN,2023-01-02T04:52:00,What is TSMC?,Tech Monitor,0.718922,0.307854,Somewhat-Bullish
1,58d72b86288d4fffa86b70eac0d65e89,AAPL,True,NaN,new,Market Value Drop,market_sentiment,"Apple Inc.'s market capitalization fell below $2 trillion for the first time since March 2021, driven by concerns over high inflation an...",risk_off,ticker,[market_uncertainty],high,high,high,"The article provides direct evidence of a significant market value drop for Apple, citing concerns over inflation and a slowing economy ...",aapl__market_sentiment__e30aecc219,2023-01-03T03:59:02,Apple’s market value drops below $2 trillion,Al Jazeera,1.000000,-0.420083,Bearish
2,aa9775754e1566cebaeea81ca19ecca4,AAPL,False,NaN,irrelevant,No material event,other,No material event identified for the target ticker.,neutral,irrelevant,[none],none,none,low,No evidence summary supplied.,NaN,2023-01-03T04:06:10,"If You Invested $100 in Berkshire Hathaway in 1965, This Is How Much You Would Have Today",The Motley Fool,0.732629,0.349630,Somewhat-Bullish
3,e94cd66eaeab77a193199183b1a77cbc,AAPL,False,NaN,irrelevant,No material event,other,No material event identified for the target ticker.,neutral,irrelevant,[none],none,none,low,No evidence summary supplied.,NaN,2023-01-03T04:06:10,"If You Invested $100 in Berkshire Hathaway in 1965, This Is How Much You Would Have Today",The Globe and Mail,0.749384,0.317527,Somewhat-Bullish
4,cd94864135b4019d7f7e9be93d4d9044,AAPL,False,NaN,irrelevant,No material event,other,No material event identified for the target ticker.,neutral,irrelevant,[none],none,none,low,No evidence summary supplied.,NaN,2023-01-03T06:49:00,Foxconn to use Nvidia chips to build self-driving vehicle platforms,Reuters,0.622889,0.106214,Neutral
5,eec1c245936a45f5e7e4f9134ac6e54d,AAPL,True,aapl__market_sentiment__e30aecc219,confirm,Market Value Drop,market_sentiment,"Apple Inc.'s market capitalization fell below $2 trillion for the first time since March 2021, driven by concerns over high inflation an...",risk_off,ticker,[market_uncertainty],high,low,high,"The article confirms the significant market value drop for Apple, citing concerns over inflation and a slowing economy as key factors.",aapl__market_sentiment__e30aecc219,2023-01-03T12:05:00,Tuesday's ETF with Unusual Volume: IWL,Nasdaq,0.784926,-0.304528,Somewhat-Bearish
6,1f45b4cbc18d012cd1fd6cdb1201c7f7,AAPL,True,aapl__market_sentiment__e30aecc219,confirm,Market Value Drop,market_sentiment,"Apple Inc.'s market capitalization fell below $2 trillion for the first time since March 2021, driven by concerns over high inflation an...",risk_off,ticker,[market_uncertainty],high,low,high,"The article confirms the significant market value drop for Apple, citing concerns over inflation and a slowing economy as key factors.",aapl__market_sentiment__e30aecc219,2023-01-03T17:38:38,Think you have it bad?,The Reformed Broker,0.714321,-0.216855,Somewhat-Bearish
7,f1a28840cb9391fc012bce188488f0e4,AAPL,True,aapl__market_sentiment__e30aecc219,confirm,Market Value Drop,market_sentiment,"Apple Inc.'s market capitalization fell below $2 trillion for the first time since March 2021, driven by concerns over high inflation an...",risk_off,ticker,[market_uncertainty],high,low,high,"The article confirms the significant market value drop for Apple, citing concerns over inflation and a slowing economy as key factors.",aapl__market_sentiment__e30aecc219,2023-01-04T09:26:00,Salesforce to lay off 10% of wo

In [16]:
response_log_df = pd.read_json(
    OUTPUT_DIR / "raw_model_responses.jsonl",
    lines=True,
)

failed = response_log_df[response_log_df["error"].notna()]

for _, row in failed.iterrows():
    print("\nARTICLE:", row["article_uid"])
    print("ERROR:", row["error"])
    print("RAW RESPONSE:")
    print(row["raw_response"][:3000])


ARTICLE: e393dc219292db3570940caf8a5e996f
ERROR: ValueError: No assessment JSON object found in the model response
RAW RESPONSE:
<event_json>
  "article_uid": "e393dc219292db3570940caf8a5e996f",
  "ticker": "AAPL",
  "relevant": false,
  "matched_event_id": null,
  "event_action": "irrelevant",
  "event_name": "No material event",
  "event_type": "other",
  "event_summary": "No material event identified for the target ticker.",
  "direction": "neutral",
  "scope": "irrelevant",
  "transmission_channels": ["none"],
  "materiality": "none",
  "novelty": "none",
  "confidence": "low",
  "evidence_summary": "The article discusses Kartoon Channel!'s expansion into new markets through partnerships with Vizio and in-flight entertainment systems, which does not have a direct causal link to Apple's operations, revenue, costs, competitive position, strategy, financing, or regulatory environment."
</event_json>

ARTICLE: 6efa566549edadd15e83cf24dfee61bc
ERROR: ValueError: No assessment JSON obje

In [ ]:
response_log_df = pd.DataFrame(
    raw_response_records,
    columns=["article_uid", "time_published", "raw_response", "error"],
)
failed_df = response_log_df.loc[response_log_df["error"].notna()].copy()

print(f"Successful responses: {len(response_log_df) - len(failed_df)}")
print(f"Failed responses: {len(failed_df)}")
if not failed_df.empty:
    display(failed_df[["article_uid", "error", "raw_response"]])


In [25]:
print("Current events")
display(events_df)

print("Event state history")
display(states_df.head(10))

print("Event evidence")
display(evidence_df.head(10))

Current events


,event_id,ticker,event_name,event_type,summary,scope,direction,transmission_channels,materiality,confidence,status,first_seen,last_updated
0,aapl__market_sentiment__e30aecc219,AAPL,Market Value Drop,market_sentiment,"Apple Inc.'s market capitalization fell below $2 trillion for the first time since March 2021, driven by concerns over high inflation an...",ticker,risk_off,[market_uncertainty],medium,high,unresolved,2023-01-03T03:59:02,2023-01-04T09:26:00
1,aapl__supply_chain__cb63444c7f,AAPL,Supply Chain Diversification,supply_chain,"Apple plans to replace Broadcom's Wi-Fi and Bluetooth chips with in-house designs, potentially impacting its supply chain.",ticker,neutral,[supply_chain],low,medium,unresolved,2023-01-04T11:52:00,2023-01-10T04:03:16
2,aapl__product_or_innovation__3e46f63a42,AAPL,AI Narration in Audiobooks,product_or_innovation,"Apple Inc. has introduced AI-narrated audiobooks through Apple Books, utilizing text-to-speech technology to make audiobook production m...",ticker,neutral,[none],low,high,unresolved,2023-01-05T03:59:02,2023-01-05T03:59:02


Event state history


,event_id,ticker,event_name,event_type,summary,scope,direction,transmission_channels,materiality,confidence,status,first_seen,last_updated,valid_from,event_action
0,aapl__market_sentiment__e30aecc219,AAPL,Market Value Drop,market_sentiment,"Apple Inc.'s market capitalization fell below $2 trillion for the first time since March 2021, driven by concerns over high inflation an...",ticker,risk_off,[market_uncertainty],medium,high,unresolved,2023-01-03T03:59:02,2023-01-03T03:59:02,2023-01-03T03:59:02,new
1,aapl__market_sentiment__e30aecc219,AAPL,Market Value Drop,market_sentiment,"Apple Inc.'s market capitalization fell below $2 trillion for the first time since March 2021, driven by concerns over high inflation an...",ticker,risk_off,[market_uncertainty],medium,high,unresolved,2023-01-03T03:59:02,2023-01-03T12:05:00,2023-01-03T12:05:00,confirm
2,aapl__market_sentiment__e30aecc219,AAPL,Market Value Drop,market_sentiment,"Apple Inc.'s market capitalization fell below $2 trillion for the first time since March 2021, driven by concerns over high inflation an...",ticker,risk_off,[market_uncertainty],medium,high,unresolved,2023-01-03T03:59:02,2023-01-03T17:38:38,2023-01-03T17:38:38,confirm
3,aapl__market_sentiment__e30aecc219,AAPL,Market Value Drop,market_sentiment,"Apple Inc.'s market capitalization fell below $2 trillion for the first time since March 2021, driven by concerns over high inflation an...",ticker,risk_off,[market_uncertainty],medium,high,unresolved,2023-01-03T03:59:02,2023-01-04T09:26:00,2023-01-04T09:26:00,confirm
4,aapl__supply_chain__cb63444c7f,AAPL,Supply Chain Diversification,supply_chain,"Apple is partnering with Luxshare Precision Industry Co Ltd for the production of premium iPhone models, aiming to diversify its supplie...",ticker,neutral,[supply_chain],medium,high,unresolved,2023-01-04T11:52:00,2023-01-04T11:52:00,2023-01-04T11:52:00,new
5,aapl__product_or_innovation__3e46f63a42,AAPL,AI Narration in Audiobooks,product_or_innovation,"Apple Inc. has introduced AI-narrated audiobooks through Apple Books, utilizing text-to-speech technology to make audiobook production m...",ticker,neutral,[none],low,high,unresolved,2023-01-05T03:59:02,2023-01-05T03:59:02,2023-01-05T03:59:02,new
6,aapl__supply_chain__cb63444c7f,AAPL,Supply Chain Diversification,supply_chain,"Apple is partnering with Luxshare Precision Industry Co Ltd for the production of premium iPhone models, aiming to diversify its supplie...",ticker,neutral,[supply_chain],medium,high,unresolved,2023-01-04T11:52:00,2023-01-06T04:31:08,2023-01-06T04:31:08,new
7,aapl__supply_chain__cb63444c7f,AAPL,Supply Chain Diversification,supply_chain,"Apple plans to replace Broadcom's Wi-Fi and Bluetooth chips with in-house designs, potentially impacting its supply chain.",ticker,neutral,[supply_chain],low,medium,unresolved,2023-01-04T11:52:00,2023-01-10T04:03:16,2023-01-10T04:03:16,new


Event evidence


,event_id,article_uid,observed_at,event_action,evidence_summary
0,aapl__market_sentiment__e30aecc219,58d72b86288d4fffa86b70eac0d65e89,2023-01-03T03:59:02,new,"The article provides direct evidence of a significant drop in Apple's market value, citing concerns over inflation and a slowing economy..."
1,aapl__market_sentiment__e30aecc219,eec1c245936a45f5e7e4f9134ac6e54d,2023-01-03T12:05:00,confirm,"The article confirms the significant drop in Apple's market value, citing concerns over inflation and a slowing economy as key factors."
2,aapl__market_sentiment__e30aecc219,1f45b4cbc18d012cd1fd6cdb1201c7f7,2023-01-03T17:38:38,confirm,"The article confirms the significant drop in Apple's market value, citing concerns over inflation and a slowing economy as key factors."
3,aapl__market_sentiment__e30aecc219,f1a28840cb9391fc012bce188488f0e4,2023-01-04T09:26:00,confirm,"The article confirms the significant drop in Apple's market value, citing concerns over inflation and a slowing economy as key factors."
4,aapl__supply_chain__cb63444c7f,6217c394b1954da3f19674ff41f5fbe9,2023-01-04T11:52:00,new,"The article details Apple's strategic move to partner with Luxshare for iPhone production, indicating efforts to mitigate supply chain r..."
5,aapl__product_or_innovation__3e46f63a42,21f04789fa15efceab5ae127c932b985,2023-01-05T03:59:02,new,"The article provides detailed information about Apple's new AI-narrated audiobooks initiative, which is a novel product offering."
6,aapl__supply_chain__cb63444c7f,ca490f805e132471c8b06298d38303f8,2023-01-06T04:31:08,new,"The article details Apple's strategic move to partner with Luxshare for iPhone production, indicating efforts to mitigate supply chain r..."
7,aapl__supply_chain__cb63444c7f,02bd82b62c7bf1d151a0808e2b11fbe3,2023-01-10T04:03:16,new,"The article mentions Apple's plan to reduce reliance on Broadcom for certain chip components, which could have implications for its supp..."


## 8. Ground event attributes into stable predicates

The event graph preserves specific facts. Predicates express reusable logical
statements.

Example:

```text
Specific event:
    "Taiwan semiconductor conflict risk"

Stable predicates:
    GeopoliticalRisk(AAPL, date)
    SupplyChainRisk(AAPL, date)
    UnresolvedMaterialRisk(AAPL, date)
```

The grounding functions below are documented and deterministic. Therefore,
`NewsRiskOff(AAPL, date) = 0.72` is not an arbitrary dictionary value: it is the
truth degree returned by a defined function applied to validated evidence.

In [1]:
def fuzzy_or(values) -> float:
    result = 0.0
    for value in values:
        value = max(0.0, min(1.0, float(value)))
        result = result + value - result * value
    return float(result)


def latest_visible_events(states: pd.DataFrame, as_of: pd.Timestamp) -> pd.DataFrame:
    if states.empty:
        return pd.DataFrame()

    history = states.copy()
    history["valid_from"] = pd.to_datetime(history["valid_from"])

    visible = (
        history.loc[history["valid_from"] <= as_of]
        .sort_values("valid_from")
        .groupby("event_id", as_index=False)
        .tail(1)
    )

    return visible.loc[visible["status"] != "resolved"].reset_index(drop=True)


def event_truths(event: pd.Series, as_of: pd.Timestamp) -> dict[str, float]:
    materiality = LEVEL_VALUE[event.materiality]
    confidence = LEVEL_VALUE[event.confidence]

    age_days = max(0, (as_of - pd.Timestamp(event.valid_from)).days)
    decay = 0.5 ** (age_days / 30.0)

    strength = materiality * confidence * decay

    return {
        "event_risk_on": strength if event.direction == "risk_on" else 0.0,
        "event_risk_off": strength if event.direction == "risk_off" else 0.0,
        "event_material": materiality * decay,
        "unresolved_material_risk": strength
        if event.direction == "risk_off"
        else 0.0,
        "geopolitical_risk": strength
        if event.event_type == "geopolitical_conflict"
        else 0.0,
        "regulatory_risk": strength
        if event.event_type == "regulatory_or_legal"
        else 0.0,
        "supply_chain_risk": strength
        if "supply_chain" in event.transmission_channels
        else 0.0,
        "demand_weakening": strength
        if "demand_weakening" in event.transmission_channels
        else 0.0,
        "input_cost_pressure": strength
        if "input_cost_pressure" in event.transmission_channels
        else 0.0,
        "operational_disruption": strength
        if "operational_disruption" in event.transmission_channels
        else 0.0,
        "market_uncertainty": strength
        if "market_uncertainty" in event.transmission_channels
        else 0.0,
    }

NameError: name 'pd' is not defined

In [ ]:
if assessments_df.empty:
    raise ValueError(
        "No valid article assessments were produced. "
        "Inspect raw_model_responses.jsonl for parsing or validation errors."
    )

dates = sorted(
    pd.to_datetime(assessments_df["time_published"])
    .dt.normalize()
    .unique()
)

daily_rows = []

for date in dates:
    as_of = pd.Timestamp(date) + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
    visible_events = latest_visible_events(states_df, as_of)

    row = {"ticker": TICKER, "date": pd.Timestamp(date)}

    event_rows = [
        event_truths(event, as_of)
        for _, event in visible_events.iterrows()
    ]

    predicate_names = sorted({name for event in event_rows for name in event})

    for predicate in predicate_names:
        row[predicate] = fuzzy_or(
            event.get(predicate, 0.0)
            for event in event_rows
        )

    row["active_event_count"] = len(visible_events)

    if visible_events.empty:
        row["risk_off_event_count"] = 0
        row["risk_on_event_count"] = 0
        row["conflicting_events"] = 0.0
    else:
        row["risk_off_event_count"] = int((visible_events["direction"] == "risk_off").sum())
        row["risk_on_event_count"] = int((visible_events["direction"] == "risk_on").sum())
        row["conflicting_events"] = min(
            row.get("event_risk_on", 0.0),
            row.get("event_risk_off", 0.0),
        )

    daily_rows.append(row)

daily_predicates = (
    pd.DataFrame(daily_rows)
    .fillna(0.0)
    .sort_values("date")
    .reset_index(drop=True)
)

display(daily_predicates)

In [ ]:
if response_log_df.empty:
    print("No model responses were recorded.")
else:
    print(response_log_df["error"].value_counts(dropna=False))
    display(response_log_df[["article_uid", "error", "raw_response"]].head())


## 9. LTN reasoning over the grounded predicates

LTN does not magically discover the meaning of predicates. The previous cells
performed their grounding.

Here, LTN fuzzy operators evaluate explicit reusable rules:

```text
NewsRiskOff ∧ NewsMaterial ∧ UnresolvedMaterialRisk → BearSupport
NewsRiskOn ∧ NewsMaterial ∧ ¬UnresolvedMaterialRisk → BullSupport
ConflictingNews ∨ (NewsMaterial ∧ ¬NewsRiskOn ∧ ¬NewsRiskOff) → NeutralSupport
```

This is a transparent inference demonstration. Later, the regime predicates or
rule weights can be made learnable using labelled regime data and LTN
satisfiability as a training objective.

In [ ]:
try:
    import ltn

    And = ltn.fuzzy_ops.AndProd()
    Or = ltn.fuzzy_ops.OrProbSum()
    Not = ltn.fuzzy_ops.NotStandard()
    Implies = ltn.fuzzy_ops.ImpliesReichenbach()
    LTN_BACKEND = "LTNtorch"
except ImportError:
    # Transparent fallback for inspecting the notebook before LTNtorch is installed.
    And = lambda x, y: x * y
    Or = lambda x, y: x + y - x * y
    Not = lambda x: 1.0 - x
    Implies = lambda x, y: 1.0 - x + x * y
    LTN_BACKEND = "transparent PyTorch fallback"

print("Logic backend:", LTN_BACKEND)


def tensor_column(frame: pd.DataFrame, name: str) -> torch.Tensor:
    if name not in frame:
        return torch.zeros(len(frame), dtype=torch.float32)
    return torch.tensor(frame[name].fillna(0.0).to_numpy(), dtype=torch.float32)


risk_off = tensor_column(daily_predicates, "news_risk_off")
risk_on = tensor_column(daily_predicates, "news_risk_on")
material = tensor_column(daily_predicates, "news_material")
unresolved = tensor_column(daily_predicates, "unresolved_material_risk")
conflicting = tensor_column(daily_predicates, "conflicting_news")
escalated = tensor_column(daily_predicates, "existing_event_escalated")
geopolitical = tensor_column(daily_predicates, "geopolitical_risk")
supply_chain = tensor_column(daily_predicates, "supply_chain_risk")

bear_rule_1 = And(And(risk_off, material), unresolved)
bear_rule_2 = And(And(escalated, geopolitical), supply_chain)
bear_support = Or(bear_rule_1, bear_rule_2)

bull_support = And(And(risk_on, material), Not(unresolved))
weak_direction = And(Not(risk_on), Not(risk_off))
neutral_support = Or(conflicting, And(material, weak_direction))

scores = torch.stack([bear_support, neutral_support, bull_support], dim=1)
scores = scores / scores.sum(dim=1, keepdim=True).clamp_min(1e-8)

ltn_results = daily_predicates[["ticker", "date"]].copy()
ltn_results[["bear_support", "neutral_support", "bull_support"]] = scores.detach().numpy()
ltn_results["detected_regime"] = [
    ["bear_drawdown", "sideways_neutral", "bull_rally"][index]
    for index in scores.argmax(dim=1).tolist()
]
display(ltn_results)

In [ ]:
# Formula-satisfaction diagnostics illustrate how LTN evaluates implications.
formula_satisfaction = pd.DataFrame(
    {
        "date": daily_predicates["date"],
        "risk_off_material_unresolved_implies_bear": Implies(
            bear_rule_1, scores[:, 0]
        ).detach().numpy(),
        "risk_on_material_implies_bull": Implies(
            And(risk_on, material), scores[:, 2]
        ).detach().numpy(),
        "conflicting_news_implies_neutral": Implies(
            conflicting, scores[:, 1]
        ).detach().numpy(),
    }
)
display(formula_satisfaction)

## 10. Inspect one prediction from evidence to rule

This final view connects:

```text
source article
→ article assessment
→ event/evidence record
→ daily predicate truth values
→ activated LTN rule
→ regime support
```

In [ ]:
if not daily_predicates.empty:
    selected_date = daily_predicates.iloc[-1]["date"]
    print("Selected date:", selected_date)

    print("\nArticles and assessments")
    display(
        assessments_df.loc[
            pd.to_datetime(assessments_df["time_published"]).dt.normalize() == selected_date
        ]
    )

    print("\nVisible event states")
    display(
        states_df.loc[
            pd.to_datetime(states_df["valid_from"])
            <= pd.Timestamp(selected_date) + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
        ]
    )

    print("\nGrounded predicates")
    display(daily_predicates.loc[daily_predicates["date"] == selected_date])

    print("\nLTN result")
    display(ltn_results.loc[ltn_results["date"] == selected_date])

## What to inspect before scaling

Do not process the full dataset yet. First inspect:

1. Does DeepSeek consistently distinguish new events from repeated reporting?
2. Do matched event IDs refer only to supplied candidate events?
3. Are event types and transmission channels stable across similar articles?
4. Do ordinal materiality and novelty assignments follow the rubric?
5. Do event-state histories contain only information available at each timestamp?
6. Are predicate truth values traceable to source evidence?
7. Are the LTN rules economically defensible?

After this works for a small January sample, increase `MAX_ARTICLES`, evaluate
multiple tickers, and replace calendar-day aggregation with an exchange-calendar
function that assigns after-hours and weekend news to the correct trading date.